# 07 · Road following — iteration 4 (final): speed modes, emergency stop, live prediction overlay

**Author:** Albert Sagré Martínez.

**Changes vs. iteration 3**
- 8 selectable speed modes; each defines a throttle band (corner floor / straight ceiling) with a finer 0.001-per-frame ramp and a tighter straight-line threshold (|steering| ≤ 0.13).
- Mode 0 and a dedicated **Stop** button force throttle to 0.
- Live camera view with the predicted target point overlaid.
- Model retrained on 6,180 frames including obstacle scenes (blue panels).

Code kept as run on the car (archival). The speed-mode logic has been extracted, refactored and unit-tested in [`src/jetracer_control`](../src/jetracer_control) — the tests prove it is behaviour-equivalent to this notebook. Known issues are listed in `docs/known-issues.md`.

# Road Following + dynamic throttling + prediction in real time

In [ ]:
import torch
from torch2trt import TRTModule

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_uni_V4_trt.pth'))

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

In [ ]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(width=224, height=224, capture_fps=15)

In [ ]:
import cv2
import traitlets
import ipywidgets
from IPython.display import display
from ipywidgets import Layout, Button, Box
import ipywidgets.widgets as widgets

state_widget = ipywidgets.ToggleButtons(options=['On', 'Off'], description='Camera', value='On')
prediction_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)

live_execution_widget = ipywidgets.VBox([
    prediction_widget,
    state_widget
],layout=Layout(align_items='center'))

network_output_slider = widgets.FloatSlider(description='Network Out', min=-1.0, max=1.0, value=0, step=0.01, orientation='horizontal', disabled=False, layout={'width': '300px'})
steering_gain_slider  = widgets.FloatSlider(description='Steering Gain', min=-1.0, max=1.0, value=-0.65, step=0.01, orientation='horizontal', layout={'width': '300px'})
steering_bias_slider  = widgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, orientation='horizontal', layout={'width': '300px'})
steering_value_slider = widgets.FloatSlider(description='Steering', min=-1.0, max=1.0, value=0, step=0.01, orientation='horizontal', disabled=False, layout={'width': '300px'})
throttle_slider = widgets.FloatSlider(description='Throttle', min=0.0, max=0.2, value=0.0, step=0.01, orientation='vertical',layout={'height': '250px'})
#  throttle_slider value should be [-1,1] set to [0,0.2] for prevention
speed_slider = widgets.IntSlider(description='Speed mode', min=0, max=8, value=0, step=1, orientation='vertical', layout={'height': '250px'})

steering_gain_link   = traitlets.link((steering_gain_slider, 'value'), (car, 'steering_gain'))
steering_offset_link = traitlets.link((steering_bias_slider, 'value'), (car, 'steering_offset'))
#steering_value_link  = traitlets.link((steering_value_slider, 'value'), (car, 'steering'))
throttle_slider_link = traitlets.link((throttle_slider, 'value'), (car, 'throttle'))

button = widgets.Button(description='Stop', button_style='danger')
out = widgets.Output()

def on_button_clicked(_):
    with out:
        speed_slider.value = 0
button.on_click(on_button_clicked)


In [ ]:
from utils import preprocess
from jetcam.utils import bgr8_to_jpeg

def update(change):
    global blocked_slider, robot
    new_image = change['new']
    
    image = preprocess(new_image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[0])
    
    network_output_slider.value = x
    steering = x * steering_gain_slider.value + steering_bias_slider.value
    if(steering<-1.0):
        steering_value_slider.value = -1.0
    elif(steering>1.0):
        steering_value_slider.value = 1.0
    else:
        steering_value_slider.value = steering 
    car.steering = x
    
    if(state_widget.value == 'On'):
        x = int(camera.width * (x / 2.0 + 0.5))
        y = int(camera.height * (y / 2.0 + 0.5))  
        prediction = new_image.copy()
        prediction = cv2.circle(prediction, (x, y), 8, (255, 0, 0), 3)
        prediction_widget.value = bgr8_to_jpeg(prediction)
    
    # Values below  may be personalized on personal preferences. Steering and speed values can be changed to fit purpose.
    steering_max = 0.13
    steering_min = -0.13
    
    if speed_slider.value == 0:
        throttle_slider.value = 0
        
    elif speed_slider.value == 1:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.14 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.12):
            throttle_slider.value += - 0.001
                    
    elif speed_slider.value == 2:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.15 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.13):
            throttle_slider.value += - 0.001
                    
    elif speed_slider.value == 3:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.15 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.14):
            throttle_slider.value += - 0.001
    
    elif speed_slider.value == 4:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.16 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.15):
            throttle_slider.value += - 0.001
    
    elif speed_slider.value == 5:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.17 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.16):
            throttle_slider.value += - 0.001
    
    elif speed_slider.value == 6:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.18 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.17):
            throttle_slider.value += - 0.001
            
    elif speed_slider.value == 7:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.19 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.18):
            throttle_slider.value += - 0.001        
    
    elif speed_slider.value == 8:
        if (steering<=steering_max and steering>=steering_min and throttle_slider.value<0.2 and throttle_slider.value>0.11):
            throttle_slider.value += 0.001
        elif((steering>steering_max or steering<steering_min) and throttle_slider.value>0.19):
            throttle_slider.value += - 0.001
    
    else:
        throttle_slider.value = throttle_slider.value     
    
update({'new': camera.value})

In [ ]:
camera.observe(update, names='value') 
camera.running = True

In [ ]:
display(
    widgets.HBox(
        [widgets.VBox([network_output_slider,
                       widgets.Label(value="X"),
                       steering_gain_slider,
                       widgets.Label(value="+"),
                       steering_bias_slider,
                       widgets.Label(value="||"), 
                       steering_value_slider,
                       button,out,], layout=Layout(align_items='center')
                     ),
         throttle_slider,
         speed_slider,
         live_execution_widget
         ]
                )
        )